# Figure 5: measured line-array reconstruction and localization

This example uses 2800/3000 Hz and configuration A (pos3). The shared command evaluates CBF, MUSIC, MVDR and SRP-PHAT for the five array reconstructions. The displayed spatial spectrum is SRP-PHAT. Fields use pressure in Pa and velocity in m/s.

Download and extract the Figure 5 data, reference predictions and FPE model into the repository. The default evaluation reads the released predictions. Set `RESAMPLE = True` to generate new fields from the 5000-step FPE checkpoint on the selected CUDA device.


In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

ROOT = Path.cwd().resolve()
while not (ROOT / 'configs/figure5_line_array.json').is_file():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Run this notebook inside the MRSM repository')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.evaluate_figure5 import load_settings, conditions, load_input, evaluate_localization

SETTINGS = load_settings()
PAIR_INDEX = 1
CONFIGURATION_INDEX = 0
RESAMPLE = False
DEVICE = 'cuda:2'
OUTPUT_DIR = ROOT / 'results/figure5_notebook'
selected_pair = '_'.join(map(str, SETTINGS['dual_frequency_hz'][PAIR_INDEX]))
selected_position = SETTINGS['source_configurations'][CONFIGURATION_INDEX]['position']
condition = next(c for c in conditions(SETTINGS)
                 if c['pair'] == selected_pair and c['position'] == selected_position)
condition['case']


In [ ]:
rows, units = evaluate_localization(ROOT, OUTPUT_DIR, SETTINGS, [condition],
                                    resample=RESAMPLE, device=DEVICE)
display(pd.DataFrame(units))


## Pressure waveforms and SRP-PHAT spectra

The sparse array uses seven physical positions. The generated field has 32 positions from x=0.48 m to x=-0.512 m. The seven observed positions coincide with generated indices 0, 5, 10, 15, 20, 25 and 30.


In [ ]:
plt.rcParams.update({'font.family': 'Arial'})
def clean_tick(value, _=None):
    return '0' if abs(value) < 1e-12 else f'{value:g}'
formatter = FuncFormatter(clean_tick)
observed = load_input(ROOT, SETTINGS, condition, 'localization')
time_ms = np.arange(observed.shape[-1]) / SETTINGS['sampling_frequency_hz'] * 1000
fig, ax = plt.subplots(figsize=(8, 3))
for i, sensor in enumerate(SETTINGS['sensor_ids']):
    ax.plot(time_ms, observed[i, 0], linewidth=0.6, label=sensor)
ax.set(xlabel='Time (ms)', ylabel='Pressure (Pa)')
ax.xaxis.set_major_formatter(formatter)
ax.yaxis.set_major_formatter(formatter)
ax.legend(title='Sensor', ncol=7, frameon=False)
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, token, title in zip(axes, ['0_seed0_SRP-PHAT', '4_seed3_SRP-PHAT'],
                            ['Seven physical AVSs', 'PI-ISS, seed 3']):
    with np.load(OUTPUT_DIR / 'localization' / condition['case'] / f'{token}.npz') as spectrum:
        image = ax.pcolormesh(spectrum['x'], spectrum['z'], spectrum['db'], shading='auto')
    truth = condition['truth']
    ax.scatter(truth[:, 0], truth[:, 1], marker='x', c='white')
    ax.set(title=title, xlabel='x (m)', ylabel='z (m)', aspect='equal')
    ax.xaxis.set_major_formatter(formatter)
    ax.yaxis.set_major_formatter(formatter)
    bar = fig.colorbar(image, ax=ax, label='Relative power (dB)', format=formatter)
fig.tight_layout()
plt.show()


## All twelve conditions and held-out sensors

Run `python scripts/evaluate_figure5.py --task all --output-dir results/figure5_evaluation` from the repository root. The localization table contains 336 per-run values and 240 condition means. The held-out calculation uses six four-channel inputs to predict one internal position, rotating through five held-out positions. It saves full and two-tone-projected waveforms and per-channel errors. The 20 channel/method groups contain 60 values each after averaging the three PI-ISS seeds.
